# Audio-Visual Contact Classification

Paper-aligned training notebook for the released processed dataset. This notebook does not modify the original dataset files.

## 1. Setup

On Kaggle, set `DATA_ROOT` to the folder that contains `audio_visual_dataset_default/` and `audio_visual_dataset_robo_default/`.

In [ ]:
from pathlib import Path
import sys

# Local project default. On Kaggle, change this to your uploaded dataset path.
DATA_ROOT = Path('/kaggle/input/contact-data/dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('../dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('dataset')

CODE_DIR = Path('/kaggle/working/coding')
if not CODE_DIR.exists():
    CODE_DIR = Path('.')
sys.path.append(str(CODE_DIR.resolve()))

print('DATA_ROOT =', DATA_ROOT.resolve())
print('CODE_DIR =', CODE_DIR.resolve())

## 2. Imports and Config

In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from config import DataConfig, TrainConfig, LABELS
from data import AudioVisualDataset, build_index, make_train_val_split
from models import AudioVisualFusionNet

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

data_cfg = DataConfig(
    data_root=DATA_ROOT,
    target_sample_rate=22000,
    audio_window_sec=0.8,
    train_crop='random',
    eval_crop='center',
    skip_missing_files=True,
)
train_cfg = TrainConfig(
    epochs=5,
    batch_size=16,
    lr=1e-3,
    num_workers=2,
    use_amp=True,
)
set_seed(train_cfg.seed)

## 3. Inspect Dataset

This step filters missing files in code. It should skip the one robot row whose image is absent from the released archive.

In [ ]:
index = build_index(data_cfg.data_root, skip_missing_files=data_cfg.skip_missing_files)
print('Valid samples:', len(index))
print('Skipped missing files:', index.attrs.get('skipped_missing_files', 0))
display(index['split_name'].value_counts())
display(index['label'].value_counts().reindex(LABELS).fillna(0).astype(int))
index.head()

## 4. DataLoaders

In [ ]:
train_df, val_df = make_train_val_split(index, train_cfg.val_size, train_cfg.seed)
train_ds = AudioVisualDataset(train_df, data_cfg, train=True)
val_ds = AudioVisualDataset(val_df, data_cfg, train=False)

train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True, num_workers=train_cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg.batch_size, shuffle=False, num_workers=train_cfg.num_workers, pin_memory=True)

batch = next(iter(train_loader))
print(batch['audio'].shape, batch['image'].shape, batch['label'].shape)

## 5. Model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AudioVisualFusionNet(num_classes=len(LABELS), pretrained_image=False).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
print('device =', device)

## 6. Training

In [ ]:
def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_acc, total_items = 0.0, 0.0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=train_cfg.use_amp and device == 'cuda')
    for batch in tqdm(loader, leave=False):
        audio = batch['audio'].to(device)
        image = batch['image'].to(device)
        labels = batch['label'].to(device)
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=train_cfg.use_amp and device == 'cuda'):
                logits = model(audio, image)
                loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy(logits.detach(), labels) * bs
        total_items += bs
    return total_loss / total_items, total_acc / total_items

best_val_acc = 0.0
out_dir = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
out_dir.mkdir(exist_ok=True, parents=True)

for epoch in range(1, train_cfg.epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, None)
    print(f'epoch={epoch} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model': model.state_dict(), 'labels': LABELS, 'data_config': data_cfg.__dict__}, out_dir / 'best_fusion_model.pt')

print('Best validation accuracy:', best_val_acc)

## 7. Notes for Paper Comparison

- This notebook uses the released processed dataset, not raw ROS bags.
- Missing files are skipped in code.
- Audio is loaded from released 44.1 kHz WAV files and resampled to 22 kHz at runtime.
- Audio windows are handled in code as 0.8 second crops.
- Mel-spectrograms are generated during loading, not saved back into the dataset.